In [ ]:
import sys, tarfile

with tarfile.open('/kaggle/input/birdclef-2026-deps-nnaudio/nnaudio.tar.gz', 'r:gz') as tar:
    tar.extractall('/tmp')
sys.path.insert(0, '/tmp/nnaudio_pkg')

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from nnAudio.features import MelSpectrogram as _MelSpectrogram

In [ ]:
# ── Transforms (vendored from birdclef_2026.data.transforms) ─────────────────

class MelSpectrogram(nn.Module):
    def __init__(self, sample_rate=32000, n_fft=2048, hop_length=512,
                 n_mels=128, fmin=20.0, fmax=16000.0):
        super().__init__()
        self.mel = _MelSpectrogram(
            sr=sample_rate, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, fmin=fmin, fmax=fmax,
            trainable_mel=False, trainable_STFT=False,
        )

    def forward(self, x):
        return self.mel(x)


class AmplitudeToDB(nn.Module):
    def __init__(self, top_db=80.0):
        super().__init__()
        self.top_db = top_db

    def forward(self, x):
        x_db = 10.0 * torch.log10(x.clamp(min=1e-9))
        max_db = x_db.amax(dim=(-2, -1), keepdim=True)
        return x_db.clamp(min=max_db - self.top_db)


class PerSampleMinMaxNorm(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x):
        min_ = x.amin(dim=(-2, -1), keepdim=True)
        max_ = x.amax(dim=(-2, -1), keepdim=True)
        return (x - min_) / (max_ - min_ + self.eps)


class Resize(nn.Module):
    def __init__(self, height, width):
        super().__init__()
        self.size = (height, width)

    def forward(self, x):
        x = x.unsqueeze(1)
        return F.interpolate(x, size=self.size, mode='bilinear', align_corners=False)


# attn_sed training params
transform = nn.Sequential(
    MelSpectrogram(sample_rate=32000, n_fft=2048, hop_length=313, n_mels=256, fmin=20.0),
    AmplitudeToDB(top_db=80.0),
    PerSampleMinMaxNorm(),
    Resize(256, 256),
)

In [ ]:
# ── Model (vendored from birdclef_2026.experiments.attn_sed.model) ────────────

class GeMPooling(nn.Module):
    def __init__(self, init_p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(init_p))
        self.eps = eps

    def forward(self, h):
        p = self.p.clamp(min=1.0)
        return h.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class AttnSEDHead(nn.Module):
    def __init__(self, num_features, num_classes, dropout=0.2):
        super().__init__()
        self.pre_fc = nn.Sequential(
            nn.Linear(num_features, num_features),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.att_fc = nn.Linear(num_features, num_classes)
        self.cls_fc = nn.Linear(num_features, num_classes)

    def forward(self, h):
        h = h.permute(0, 2, 1)
        h = self.pre_fc(h)
        att_w = F.softmax(torch.tanh(self.att_fc(h)), dim=1)
        timewise = self.cls_fc(h)
        logits = (att_w * timewise).sum(dim=1)
        return logits, timewise


class AttnSEDModel(nn.Module):
    def __init__(self, n_classes, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            'hgnetv2_b0.ssld_stage2_ft_in1k',
            pretrained=False,
            in_chans=1,
            num_classes=0,
            global_pool='',
        )
        dummy = torch.zeros(1, 1, 256, 256)
        with torch.no_grad():
            num_features = self.backbone(dummy).shape[1]
        self.gem = GeMPooling()
        self.head = AttnSEDHead(num_features, n_classes, dropout=dropout)

    def forward(self, x):
        h = self.backbone(x)
        h = self.gem(h)
        logits, _ = self.head(h)
        return logits

In [ ]:
COMPETITION_DIR     = '/kaggle/input/competitions/birdclef-2026'
CHECKPOINT_PATH     = '/kaggle/input/datasets/joshgreen/birdclef-2026-checkpoint/checkpoint.pt'
TAXONOMY_PATH       = os.path.join(COMPETITION_DIR, 'taxonomy.csv')
TEST_SOUNDSCAPE_DIR = os.path.join(COMPETITION_DIR, 'test_soundscapes')

SAMPLE_RATE   = 32000
CHUNK_SECONDS = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS

taxonomy = pd.read_csv(TAXONOMY_PATH)
class_labels = sorted(taxonomy['primary_label'].astype(str).tolist())
n_classes = len(class_labels)
print(f'{n_classes} classes')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transform = transform.to(device).eval()

model = AttnSEDModel(n_classes=n_classes, dropout=0.2).to(device)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from {CHECKPOINT_PATH}')

In [ ]:
test_soundscapes = sorted(f for f in os.listdir(TEST_SOUNDSCAPE_DIR) if f.endswith('.ogg'))
print(f'{len(test_soundscapes)} test soundscapes')

rows = []
for filename in test_soundscapes:
    filepath = os.path.join(TEST_SOUNDSCAPE_DIR, filename)
    sig, rate = sf.read(filepath, dtype='float32')
    if rate != SAMPLE_RATE:
        raise ValueError(f'{filename}: expected {SAMPLE_RATE}Hz, got {rate}Hz')
    soundscape_id = filename.split('.')[0]

    for i in range(0, len(sig), CHUNK_SAMPLES):
        chunk = sig[i:i + CHUNK_SAMPLES]
        if len(chunk) < CHUNK_SAMPLES:
            chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))

        waveform = torch.from_numpy(chunk).unsqueeze(0).to(device)
        with torch.no_grad():
            image = transform(waveform)
            logits = model(image)
            scores = torch.sigmoid(logits).squeeze(0).cpu().numpy()

        end_time = (i // CHUNK_SAMPLES + 1) * CHUNK_SECONDS
        row_id = f'{soundscape_id}_{end_time}'
        rows.append([row_id] + scores.tolist())

submission = pd.DataFrame(rows, columns=['row_id'] + class_labels)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission)} rows x {len(submission.columns)} columns')
submission.head()